# ADHD-200 benchmark: repeated site-stratified CV

This notebook is the presentation-friendly benchmark. It compares fMRI representations with demographic and motion/QC baselines using repeated subject-level cross-validation. Every outer fold contains samples from all available sites and both diagnoses.

The completed leave-one-site-out experiments remain a separate domain-shift stress test. Research use only; this is not a clinical diagnostic model.

In [ ]:
from google.colab import drive
drive.mount('<DRIVE_MOUNT>')

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
warnings.filterwarnings('ignore')

ROOT=Path('<DATA_DIR>')
BRAINLM=ROOT/'fmri'/'brainlm_a424'
OUT=ROOT/'fmri'/'benchmark_site_stratified_cv'
OUT.mkdir(parents=True,exist_ok=True)
BENCH=ROOT/'fmri'/'strict_loso_benchmark'

cohort=pd.read_csv(BENCH/'locked_primary_cohort_409_with_motion.csv',dtype={'subject_id':str})
bank=np.load(BENCH/'a424_feature_bank.npz',allow_pickle=True)
assert np.array_equal(bank['subject_id'].astype(str),cohort.subject_id.astype(str).to_numpy())
emb_meta=pd.read_csv(BRAINLM/'brainlm_subject_embedding_metadata.csv',dtype={'subject_id':str})
emb=np.load(BRAINLM/'brainlm_subject_embeddings.npy')
emap={s:i for i,s in enumerate(emb_meta.subject_id.astype(str))}
X_brainlm=np.stack([emb[emap[s]] for s in cohort.subject_id.astype(str)])
X_fc=bank['fc_summary'].astype(np.float32)
X_spectral=bank['spectral'].astype(np.float32)
X_conf=cohort[['age','sex_male','mean_fd','max_fd','pct_fd_gt_0p2','mean_dvars','n_volumes','qc_rest']].to_numpy(np.float32)
X_age_sex=cohort[['age','sex_male']].to_numpy(np.float32)
y=cohort.label.to_numpy(int)
site=cohort.site.astype(str).to_numpy()
strata=np.char.add(np.char.add(site,'__'),y.astype(str))
display(cohort.groupby(['site','label']).size().rename('n').reset_index())
print('Subjects:',len(cohort),'Sites:',cohort.site.nunique())

## Leakage-safe repeated cross-validation

Four folds are used because the smallest site-by-label cell contains four subjects. Repeating the split across five seeds provides 20 outer evaluations. Logistic-regression regularization is selected in an inner three-fold loop. Imputation and scaling are refit inside every inner and outer training fold.

In [ ]:
def make_model(C):
    return make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        LogisticRegression(C=C,class_weight='balanced',max_iter=3000,solver='liblinear')
    )

def repeated_nested_cv(X,name,Cs=(0.01,0.1,1.0),outer_splits=4,repeats=5):
    outer=RepeatedStratifiedKFold(n_splits=outer_splits,n_repeats=repeats,random_state=42)
    fold_rows=[]; prediction_rows=[]
    for fold,(tr,te) in enumerate(outer.split(X,strata),1):
        repeat=(fold-1)//outer_splits
        inner=StratifiedKFold(n_splits=3,shuffle=True,random_state=1000+fold)
        scores={C:[] for C in Cs}
        for itr,iva in inner.split(X[tr],strata[tr]):
            for C in Cs:
                m=make_model(C).fit(X[tr][itr],y[tr][itr])
                scores[C].append(roc_auc_score(y[tr][iva],m.predict_proba(X[tr][iva])[:,1]))
        best=max(Cs,key=lambda C:np.mean(scores[C]))
        model=make_model(best).fit(X[tr],y[tr])
        prob=model.predict_proba(X[te])[:,1]
        pred=(prob>=0.5).astype(int)
        fold_rows.append({'model':name,'repeat':repeat,'fold':fold,'n_test':len(te),'C':best,
                          'auc':roc_auc_score(y[te],prob),
                          'average_precision':average_precision_score(y[te],prob),
                          'balanced_accuracy':balanced_accuracy_score(y[te],pred)})
        prediction_rows.extend({'model':name,'repeat':repeat,'fold':fold,
                                'subject_id':cohort.subject_id.iloc[i],
                                'site':site[i],'y':y[i],'prob':p}
                               for i,p in zip(te,prob))
    return pd.DataFrame(fold_rows),pd.DataFrame(prediction_rows)


In [ ]:
feature_sets={
    'age_sex':X_age_sex,
    'motion_qc':X_conf[:,2:],
    'age_sex_motion_qc':X_conf,
    'fc_roi_summary':X_fc,
    'spectral':X_spectral,
    'brainlm_frozen':X_brainlm,
    'fc_plus_brainlm':np.c_[X_fc,X_brainlm],
    'confounds_plus_brainlm':np.c_[X_conf,X_brainlm],
}

folds=[]; predictions=[]
for name,X in feature_sets.items():
    print('Running',name,flush=True)
    f,p=repeated_nested_cv(np.asarray(X,dtype=np.float32),name)
    folds.append(f); predictions.append(p)
folds=pd.concat(folds,ignore_index=True)
predictions=pd.concat(predictions,ignore_index=True)
summary=(folds.groupby('model').agg(n_evaluations=('auc','size'),
          mean_auc=('auc','mean'),sd_auc=('auc','std'),
          mean_ap=('average_precision','mean'),
          mean_balanced_accuracy=('balanced_accuracy','mean')).reset_index()
          .sort_values('mean_auc',ascending=False))
folds.to_csv(OUT/'benchmark_cv_fold_metrics.csv',index=False)
predictions.to_csv(OUT/'benchmark_cv_predictions.csv',index=False)
summary.to_csv(OUT/'benchmark_cv_summary.csv',index=False)
display(summary.style.format({'mean_auc':'{:.3f}','sd_auc':'{:.3f}',
                              'mean_ap':'{:.3f}','mean_balanced_accuracy':'{:.3f}'}))

In [ ]:
plot=summary.sort_values('mean_auc')
fig,ax=plt.subplots(figsize=(9,5))
ax.barh(plot.model,plot.mean_auc,xerr=plot.sd_auc,color=['#4C78A8']*len(plot),alpha=.9)
ax.axvline(.5,color='black',ls='--',lw=1,label='chance AUC')
ax.set(xlabel='Repeated site-stratified CV AUC',ylabel='',xlim=(0.35,0.85),
       title='ADHD-200 model comparison')
ax.legend(loc='lower right'); fig.tight_layout()
fig.savefig(OUT/'benchmark_model_comparison.png',dpi=180,bbox_inches='tight')
plt.show()

subject_mean=(predictions.groupby(['model','subject_id','site','y'],as_index=False).prob.mean())
site_rows=[]
for (model_name,site_name),d in subject_mean.groupby(['model','site']):
    if d.y.nunique()==2:
        site_rows.append({'model':model_name,'site':site_name,'n':len(d),
                          'auc':roc_auc_score(d.y,d.prob)})
site_summary=pd.DataFrame(site_rows)
site_summary.to_csv(OUT/'benchmark_cv_site_metrics.csv',index=False)
display(site_summary.pivot(index='site',columns='model',values='auc').round(3))

## How to present the result

Use the repeated site-stratified score as the main within-ADHD-200 result. Compare image-only and combined models against `age_sex_motion_qc`. Keep the existing nested LOSO table in the robustness section to demonstrate awareness of domain shift. A higher mixed-site result must not be described as performance on an unseen hospital.